# 1 つの原文から多様な訳を複数出す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/23_machine_translation/diverse_translation.ipynb)

NLP クックブック 23-16 ／ 機械翻訳

1 つの原文に対して、言い回しの違う訳を複数出したい。ビームサーチの上位候補が似たものばかりになる問題にも対処したい。

In [1]:
!pip install -q sentencepiece sacremoses sacrebleu

In [2]:
import itertools
import sacrebleu
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(name, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(name).eval()
JA = tokenizer.convert_tokens_to_ids("jpn_Jpan")

@torch.no_grad()
def generate(text, n=5, **kwargs):
    enc = tokenizer(text, return_tensors="pt")
    out = model.generate(**enc, max_new_tokens=64, num_return_sequences=n, forced_bos_token_id=JA, **kwargs)
    return tokenizer.batch_decode(out, skip_special_tokens=True)

def diversity(candidates):
    """候補どうしの似ていなさ: 1 − (候補の組の文字 n-gram 一致度の平均)。大きいほど多様"""
    pairs = list(itertools.combinations(candidates, 2))
    return 1 - sum(sacrebleu.sentence_chrf(a, [b]).score for a, b in pairs) / len(pairs) / 100

text = "I am sorry, but I cannot attend the meeting tomorrow because I have a doctor's appointment."
methods = {
    "ビームサーチの上位 5 件": dict(num_beams=5),
    "ビーム幅を広げる (20)": dict(num_beams=20),
    "サンプリング (温度 1.0)": dict(do_sample=True, top_k=0, temperature=1.0),
    "サンプリング (top-p 0.9)": dict(do_sample=True, top_p=0.9),
    "サンプリング (温度 0.7)": dict(do_sample=True, top_k=0, temperature=0.7),
}
torch.manual_seed(0)
for name_, kwargs in methods.items():
    cands = generate(text, **kwargs)
    print(f"--- {name_} (多様性 {diversity(cands):.2f}, 異なる訳 {len(set(cands))} 種類)")
    for c in cands:
        print("   ", c)

--- ビームサーチの上位 5 件 (多様性 0.20, 異なる訳 5 種類)
    申し訳ありませんが,私は医者に予約があるので 明日の会議に出席できません.
    申し訳ありませんが,私は医者に予約があるので,明日会議に出席できません.
    申し訳ありませんが,私は医者に予約があるので,明日の会議に出席できません.
    申し訳ありませんが,私は医者に予約があるので 明日の会議に出席することはできません.
    申し訳ありませんが,私は医者に予約があるので,明日会議に出席することはできません.
--- ビーム幅を広げる (20) (多様性 0.21, 異なる訳 5 種類)
    申し訳ありませんが,医者に予約があるので 明日の会議に出席できません.
    申し訳ありませんが,私は医者に予約があるので 明日の会議に出席できません.
    申し訳ありませんが,医者に診察があるので 明日の会議に出席できません.
    申し訳ありませんが,私は医者に予約があるので,明日会議に出席できません.
    申し訳ありませんが,私は医者に予約があるので,明日の会議に出席できません.
--- サンプリング (温度 1.0) (多様性 0.87, 異なる訳 5 種類)
    申し訳ないのですがudž明日の集まりには憑依できません さあ医 ⵣに行きたいってことあります
    私は残念ですが 私は明日の診療所に来ることができません 何か医者まで行く日立証があるからですregional conference は 私は帰国しました
    申し訳ないのですが,omvuに診療があるので 明日の会に出席しなくていい.
    申し訳ありませんが診療のためおも眠室へ来 detyrしているから 明日の集会には開放することは出来ません.
    私は 죄송ですが,医者に行く予定だから明日の食事会に参加できない/
--- サンプリング (top-p 0.9) (多様性 0.81, 異なる訳 5 種類)
    ごめんなさい,明日会場には来られないのです 診察予定のためです.
    私はすいませんが,私は明日の会合に来る事ができない.私の日程が医者へがあるからです.
    申し訳ない. 私は診察予定から 明日の会議に来ることはできません.
    私は申し訳ないが,医者に診療

In [3]:
# 訳の先頭を指定して、違う出だしの訳を作る (デコーダの入力を途中まで与える)
@torch.no_grad()
def translate_with_prefix(text, prefix):
    enc = tokenizer(text, return_tensors="pt")
    prefix_ids = tokenizer(prefix, add_special_tokens=False, return_tensors="pt")["input_ids"]
    start = torch.tensor([[model.config.decoder_start_token_id, JA]])          # NLLB のデコーダは「開始記号 + 言語タグ」から始まる
    out = model.generate(**enc, decoder_input_ids=torch.cat([start, prefix_ids], dim=1), num_beams=4, max_new_tokens=64)
    return tokenizer.decode(out[0], skip_special_tokens=True)

for prefix in ["申し訳", "すみません", "明日", "病院", "医者"]:
    print(f"[{prefix}] {translate_with_prefix(text, prefix)}")

[申し訳] 申し訳ないが,私は明日会議に出席できない. 医者に予約があるから.
[すみません] すみません 明日の会議に出席できないので 医師の診察があります.
[明日] 明日会議に出席できないのは医者に予約があるから.
[病院] 病院に行く予定があるから 明日の会議に出席できない.
[医者] 医者への診察があるから 明日の会議に出席できない.


In [4]:
# 多様性と質の両方を測る: 多様にするほど、訳の質 (参照訳との一致) は下がりやすい
from datasets import load_dataset

ds = load_dataset("Helsinki-NLP/opus-100", "en-ja", split="test")
sample = [(ex["translation"]["en"], ex["translation"]["ja"]) for ex in ds if 5 <= len(ex["translation"]["en"].split()) <= 15][:30]
torch.manual_seed(0)
for name_, kwargs in methods.items():
    div, quality, best = [], [], []
    for en, ja in sample:
        cands = generate(en, **kwargs)
        scores = [sacrebleu.sentence_chrf(c, [ja]).score for c in cands]
        div.append(diversity(cands)); quality.append(sum(scores) / len(scores)); best.append(max(scores))
    print(f"{name_}: 多様性 {sum(div) / len(div):.2f} / 平均の質 (chrF) {sum(quality) / len(quality):.1f} / 最も良い候補の質 {sum(best) / len(best):.1f}")

ビームサーチの上位 5 件: 多様性 0.37 / 平均の質 (chrF) 12.0 / 最も良い候補の質 15.4
ビーム幅を広げる (20): 多様性 0.37 / 平均の質 (chrF) 12.0 / 最も良い候補の質 15.2
サンプリング (温度 1.0): 多様性 0.94 / 平均の質 (chrF) 5.9 / 最も良い候補の質 11.5
サンプリング (top-p 0.9): 多様性 0.85 / 平均の質 (chrF) 9.1 / 最も良い候補の質 14.6
サンプリング (温度 0.7): 多様性 0.82 / 平均の質 (chrF) 9.9 / 最も良い候補の質 16.8


- ビームサーチの上位候補は、文末や助詞が少し違うだけの、ほとんど同じ訳になりやすい。ビーム幅を広げても、多様性はあまり増えない。
- サンプリングは多様な訳を出すが、温度が高いと誤訳や不自然な訳が混ざる。NLLB のような多言語モデルでは、温度 1.0 のサンプリングで他の言語の文字 (韓国語、ラテン文字の単語など) まで混ざった。温度や top-p で、多様性と質の釣り合いを調整する。
- 訳の出だしを指定すると、語順や構文の違う訳を狙って作れる。翻訳の候補を人に提示する場面 (翻訳支援) で便利。
- 複数の候補のうち最も良いものの質は、ほどほどに多様な設定 (ここでは温度 0.7) で最も高く、多様にしすぎると下がった。候補を出してから品質推定やリランキングで 1 つを選ぶ、という使い方ができる。
- 順方向と逆方向 (文末から生成する) のモデルの出力を組み合わせる、言い換えモデルを通す、といった方法でも多様な訳が得られる。